In [ ]:
from utils.util_octree_stuff import *
from utils.util_sample_stuff import *
from utils.util_visual_stuff import *
from loss.octree_losses import *
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from dataset.voxel_dataset import VoxelPatchDataset, get_dataloader

In [ ]:
DEPTH_IN = 6
DEPTH_STOP = 6 # can try 5 
LATENT_DIM = 16
FULL_DEPTH = 3
TOTAL_SEMANTIC_CLASSES = 92

PATCH_DIR = "data/patches_128x128x64_s16"
loader = get_dataloader(PATCH_DIR, batch_size=1)
output_dir = './split_outputs'
cnt = 0

pbar = tqdm(loader, desc=f"Epoch {epoch}")
for vox, _ in pbar:
    vox = vox.type(torch.LongTensor).cuda()

    # === preprocess voxel/octree as before ===
    patch = voxel_to_patch(vox, patch_size=2)
    non_empty_mask = get_non_empty_mask(vox[0], 2)

    points = voxel_grid_to_points(non_empty_mask)
    octree_test = points2octree(points, depth=6, full_depth=4)
    octree_test = octree_test.cuda()



    split = octree2split_small(octree_test, 4)
    # === Continuous timestep sampling ===
    split_big = split2splitbig(split)
    
    
    
    save_path = os.path.join(output_dir, f"split_big_{cnt:04d}.pt")
    torch.save(split_big, save_path)
    cnt += 1

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )
    def forward(self, x):
        return self.block(x)

# -----------------------------
# Spatial-latent 16×16×16 VAE
# -----------------------------
class VoxelVAE(nn.Module):
    """
    Keeps spatial size (D,H,W) = (16,16,16).
    Encodes to latent with z_channels channels: z ~ N(mu, sigma) at each voxel.
    """
    def __init__(self, z_channels=4, base=32, in_ch=1):
        super().__init__()
        self.z_channels = z_channels

        # Encoder: [1,16,16,16] -> [base] -> [2*z_channels] (mu & logvar)
        self.enc_stem = ConvBlock(in_ch, base)         # [B, base, 16,16,16]
        self.enc_mid  = ConvBlock(base, base)          # [B, base, 16,16,16]
        self.to_mu    = nn.Conv3d(base, z_channels, kernel_size=3, padding=1)
        self.to_logv  = nn.Conv3d(base, z_channels, kernel_size=3, padding=1)

        # Decoder: z -> [base] -> [1]
        self.dec_in   = nn.Conv3d(z_channels, base, kernel_size=3, padding=1)
        self.dec_mid  = ConvBlock(base, base)
        self.dec_out  = nn.Conv3d(base, 1, kernel_size=3, padding=1)  # logits

    def encode(self, x):
        h = self.enc_stem(x)
        h = self.enc_mid(h)
        mu    = self.to_mu(h)
        logvar = self.to_logv(h)
        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.dec_in(z)
        h = self.dec_mid(h)
        logits = self.dec_out(h)  # logits for BCE
        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar, z


In [ ]:
structure_vae = VoxelVAE(z_channels=2, base=32, in_ch=1).cuda()  # must match your training config
structure_vae.load_state_dict(torch.load("vae_trained_32_32_32_2.pt", map_location=device))  # path to your checkpoint

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Sinusoidal timestep embedding ---
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(
            -torch.arange(half, device=device) * (torch.log(torch.tensor(10000.0)) / (half - 1))
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

# --- Residual Block with FiLM time conditioning ---
class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim=None):
        super().__init__()
        self.norm1 = nn.InstanceNorm3d(in_ch, affine=True)
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.InstanceNorm3d(out_ch, affine=True)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        self.skip = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

        if time_emb_dim is not None:
            self.time_mlp = nn.Sequential(
                nn.SiLU(),
                nn.Linear(time_emb_dim, out_ch)
            )
        else:
            self.time_mlp = None

    def forward(self, x, t_emb=None):
        h = self.conv1(self.act(self.norm1(x)))
        if self.time_mlp is not None and t_emb is not None:
            h = h + self.time_mlp(t_emb).view(t_emb.size(0), -1, 1, 1, 1)
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)

# --- UNet3D for diffusion ---
class UNet3DModel(nn.Module):
    def __init__(self, in_ch, base_ch=64, time_emb_dim=128):
        super().__init__()
        # timestep embedding MLP
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim*4),
            nn.SiLU(),
            nn.Linear(time_emb_dim*4, time_emb_dim)
        )

        # Encoder
        self.enc1 = ResBlock3D(in_ch, base_ch, time_emb_dim)
        self.enc2 = ResBlock3D(base_ch, base_ch*2, time_emb_dim)
        self.enc3 = ResBlock3D(base_ch*2, base_ch*4, time_emb_dim)
        self.pool = nn.MaxPool3d(2)

        # Bottleneck
        self.mid = ResBlock3D(base_ch*4, base_ch*8, time_emb_dim)

        # Decoder
        self.up3 = nn.ConvTranspose3d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = ResBlock3D(base_ch*8, base_ch*4, time_emb_dim)

        self.up2 = nn.ConvTranspose3d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = ResBlock3D(base_ch*4, base_ch*2, time_emb_dim)

        self.up1 = nn.ConvTranspose3d(base_ch*2, base_ch, 2, stride=2)
        self.dec1 = ResBlock3D(base_ch*2, base_ch, time_emb_dim)

        # Output
        self.out_norm = nn.InstanceNorm3d(base_ch, affine=True)
        self.out_conv = nn.Conv3d(base_ch, in_ch, 3, padding=1)

    def forward(self, x, t):
        # embed timestep
        t_emb = self.time_mlp(t)

        # Encoder
        e1 = self.enc1(x, t_emb)             # [B,64,32,32,32]
        e2 = self.enc2(self.pool(e1), t_emb) # [B,128,16,16,16]
        e3 = self.enc3(self.pool(e2), t_emb) # [B,256,8,8,8]

        # Bottleneck
        m = self.mid(self.pool(e3), t_emb)   # [B,512,4,4,4]

        # Decoder
        d3 = self.up3(m)                     # [B,256,8,8,8]
        d3 = self.dec3(torch.cat([d3, e3], dim=1), t_emb)

        d2 = self.up2(d3)                    # [B,128,16,16,16]
        d2 = self.dec2(torch.cat([d2, e2], dim=1), t_emb)

        d1 = self.up1(d2)                    # [B,64,32,32,32]
        d1 = self.dec1(torch.cat([d1, e1], dim=1), t_emb)

        out = self.out_conv(F.silu(self.out_norm(d1)))
        return out


In [ ]:
zC = 2   # number of channels in your latent
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
diffusion_model = UNet3DModel(in_ch=zC, base_ch=64, time_emb_dim=128).to(device)

diffusion_model.load_state_dict(torch.load("structure_latent_diffusion.pt", map_location=device))

In [ ]:
def make_beta_schedule(T=1000, beta_start=1e-4, beta_end=2e-2):
    betas = torch.linspace(beta_start, beta_end, T, dtype=torch.float32)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return betas, alphas, alphas_cumprod

def q_sample(x0, t, noise, alphas_cumprod):
    # x_t = sqrt(ā_t) x0 + sqrt(1-ā_t) noise
    a_bar = alphas_cumprod[t].view(-1,1,1,1,1)
    return torch.sqrt(a_bar)*x0 + torch.sqrt(1.0 - a_bar)*noise
T = 1000
betas, alphas, alphas_cumprod = make_beta_schedule(T)
alphas_cumprod = alphas_cumprod.to(device)

In [ ]:

T = 1000
betas, alphas, alphas_cumprod = make_beta_schedule(T)
betas = betas.to(device)
alphas = alphas.to(device)
alphas_cumprod = alphas_cumprod.to(device)

# === Compute ᾱ_{t-1} ===
alphas_cumprod_prev = torch.cat([
    torch.ones(1, dtype=alphas_cumprod.dtype, device=device),
    alphas_cumprod[:-1]
], dim=0)

diffusion_model.eval()
z_samp = sample_latents_ddpm(n=1, T=1000)  # generate one latent
logits = structure_vae.decode(z_samp)
occ = (torch.sigmoid(logits) > 0.5).float()*2 - 1
octree_test = split2octree_small(splitbig2split(occ), 6, 4)

octree_recon = vae.create_child_octree(octree_test).cuda()
doctree_recon = DualOctree(octree_recon)
doctree_recon.post_processing_for_docnn()
z_rand = torch.randn(doctree_recon.total_num, LATENT_DIM)
z_T = z_rand.cuda()
z_sampled = ddim_sample_logsnr_cosine(
    z_T=z_T,
    model=unet_model,
    doctree=doctree_recon,
    S=40
    
)
output = vae.decode_code(z_sampled.cuda(), doctree_recon, update_octree=False, pos=None)

recon_voxel = reconstruct_voxel_from_patch(
    sem_voxs=output['sem_voxs'],
    octree=output['octree_out'],
    depth=6,
    shape=(1, 128, 128, 64),
    patch_size=2
)
visualize_voxel_labels(recon_voxel[0])